# Money brain on a Colab A100

Runs the whole system (`docs/COLAB.md`).
- **GPU:** only the language model (Qwen3-32B, served by vLLM) reads and explains.
- **CPU:** the memory bank, the evidence engine, your chain engine and the scorekeeper, as on any machine.

Paper only: nothing here places orders. Public data only (`CLAUDE.md`).

**Before you start:**
- Runtime → Change runtime type → **A100 GPU**.
  - **Building the full bank (`BANK = "full"`, section 7)?** Pick a **CPU** runtime for that run instead. The build needs no GPU, so the notebook skips the model, and an idle A100 isn't billed for an hour or two. Switch to the A100 once the bank is saved to Drive (section 12).
- Optional Colab secrets (the key icon on the left):
  - `SEC_USER_AGENT` (`Your Name your@email.com`): the SEC requires it.
  - `GITHUB_TOKEN`: only if the repo is private, or to push results.

## 1. GPU and model choice

In [ ]:
import subprocess, json, shutil, time
mem_gb = 0
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
    mem_gb = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                                capture_output=True, text=True).stdout.split()[0]) / 1024
# 80 GB: the full model (about 65 GB to download). 40 GB: the official 4-bit build (AWQ, about 20 GB), close in quality
# and three times quicker to download each session. Set MODEL by hand to override.
MODEL = None if mem_gb < 30 else "Qwen/Qwen3-32B" if mem_gb > 70 else "Qwen/Qwen3-32B-AWQ"
MAX_LEN = 32768
print(f"{mem_gb:.0f} GB GPU -> {MODEL}" if MODEL else "No GPU: a CPU-only run (build and refresh the bank; no model)")

## 2. Storage: the bank lives on Drive, work happens on local disk
Colab wipes the machine when the session ends.
- The bank is copied from Drive to `/content/var` at the start (fast local disk).
- It is copied back at the end, in section 12.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_VAR = "/content/drive/MyDrive/money/var"
LOCAL_VAR = "/content/var"
os.makedirs(DRIVE_VAR, exist_ok=True); os.makedirs(LOCAL_VAR, exist_ok=True)
!rsync -a --info=progress2 --exclude "/cache/*.gz" --exclude "/cache/*.json" "{DRIVE_VAR}/" "{LOCAL_VAR}/"
# An archive made on another machine (tar czf bank.tgz var/bank var/series) and uploaded to MyDrive/money/:
if not os.path.exists(f"{LOCAL_VAR}/bank") and os.path.exists("/content/drive/MyDrive/money/bank.tgz"):
    !tar xzf /content/drive/MyDrive/money/bank.tgz -C /content && echo restored from bank.tgz
!du -sh {LOCAL_VAR} 2>/dev/null

## 3. The code

In [ ]:
from google.colab import userdata
def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
tok = secret("GITHUB_TOKEN")
url = f"https://{tok}@github.com/nikku03/money" if tok else "https://github.com/nikku03/money"
if not os.path.exists("/content/money"):
    !git clone -q -b claude/upbeat-turing-z9jk2b {url} /content/money
else:
    !git -C /content/money pull -q
%cd /content/money
!git log --oneline -1
!pip install -q -e .

## 4. The language model (vLLM), in the background
Skipped on a CPU runtime. Three steps: install vLLM, download the weights (with a progress bar), start the server.

In [ ]:
if MODEL:
    !pip install -q vllm

In [ ]:
if MODEL:                                # Colab wipes the disk, so this runs every session
    from huggingface_hub import snapshot_download
    print(snapshot_download(MODEL))

In [ ]:
import urllib.request
ready = False
if MODEL and (globals().get("vllm") is None or vllm.poll() is not None):   # re-running this cell only waits again
    vllm = subprocess.Popen(["vllm", "serve", MODEL, "--max-model-len", str(MAX_LEN), "--gpu-memory-utilization", "0.95",
                             "--enable-auto-tool-choice", "--tool-call-parser", "hermes", "--port", "8000"],
                            stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
if MODEL:
    for i in range(120):                 # loading onto the GPU takes about 3-10 minutes
        if vllm.poll() is not None:
            print("the model server stopped:"); break
        try:
            served = json.load(urllib.request.urlopen("http://127.0.0.1:8000/v1/models"))["data"][0]["id"]
            print("model ready:", served); ready = True; break
        except Exception:
            time.sleep(10)
    else:
        print("still loading after 20 minutes (re-run this cell to wait more); the log so far:")
    if not ready:
        !tail -25 /content/vllm.log

## 5. Settings the code reads

In [ ]:
os.environ.update({
    "MONEY_HOME": LOCAL_VAR,
    "SEC_USER_AGENT": secret("SEC_USER_AGENT") or "money-brain research you@example.com",  # put your own in the secret
    "BRAIN_LLM_URL": "http://127.0.0.1:8000/v1",
    "BRAIN_LLM_MODEL": MODEL,
    "BRAIN_LLM_CTX": str(MAX_LEN),
    "BRAIN_LLM_TIMEOUT": "600",
})
from brain import llm
print(llm.available())

## 6. Check the code and the bank (optional, about 3 minutes)

In [ ]:
!MONEY_HOME=/tmp/testhome python -m unittest discover -s tests -t . 2>&1 | tail -3
!python -m brain check | tail -2

## 7. Data: a small bank to test the model, or the full one
- **`BANK = "small"`** (the default, about 12 minutes the first time, 470 MB): 65 stocks, plus SPY, world markets, the Treasury curve and the last 120 days of insider trades. That's enough to test the model reading, deciding and forecasting, and to use the terminal.
  - The stocks are the 60 that the decision rules rank against (`universe/us_large.json`), plus oil, airlines and homebuilders for the chain engine's links.
  - The forecast model then learns from these stocks only. Its card says so; its numbers only show the pipeline working.
  - The research tests (`brain research`) need the full bank.
- **`BANK = "full"`**: all 741 stocks and the insider history since 2010. About 1–2 hours plus the insider history, best on a CPU runtime.

Running this cell again later is the daily refresh: it only adds what is new.

In [ ]:
BANK = "small"
SMALL = (json.load(open("universe/us_large.json"))["tickers"]   # the 60 the decision rules rank against
         + ["XOM", "DAL", "UAL", "LEN", "DHI"])                    # oil, airlines, homebuilders: the chain's links
WATCHLIST = ["AAPL", "NVDA", "MSFT"]
WL = " ".join(WATCHLIST)
ST = " ".join(dict.fromkeys(SMALL + WATCHLIST))
if BANK == "full":
    !python -m brain ingest-universe --broad
    !python -m brain insiders
    !python -m brain ingest {WL}                     # the watchlist's headlines too
else:
    !python -m brain ingest {ST}                     # SEC statements, filings, prices, headlines
    !python -m brain insiders {ST} --live-only       # insider trades of the last 120 days
!python -m brain ingest SPY --skip sec filings news  # the benchmark: prices only
!python -m brain markets
!python -m brain macro
!python -m brain alerts check

## 8. Your chain engine: cause → effect

In [ ]:
!python -m brain chain train
!python -m brain chain follow oil_price_up
!python -m brain chain follow inflation_up --hops 5
!python -m brain chain links                 # each link's test result, or what would measure it

**Links from the model alone.** The model gets only the state names and writes its own cause → effect links. The report compares them with the textbook, runs them through the chain engine, and sets them beside the market tests. Needs the model from section 4; about 2 minutes on the A100.

In [ ]:
!python -m brain chain distill

**The web gateway, no model.** The reasoning engine picks what to read from the bank's gaps: the stories behind each headline, and the newsroom feeds in `knowledge/web_sources.json`. A real browser (browser-use, no agent) renders the pages, and fixed rules keep the facts. Runs on the CPU, so it works without the GPU too.

In [ ]:
!pip install -q "browser-use>=0.13,<0.14" playwright
!playwright install --with-deps chromium > /dev/null   # the browser itself; without it the gateway reads over plain HTTP
!python -m brain web plan {WL}
!python -m brain web run {WL}

**Small and micro caps: event tests (optional, CPU, many hours).** Builds the two universes (SEC rows 1501–3000 and 3001–4500 with 10-K XBRL and prices since 2010), then reads every earnings release's opening from SEC EDGAR at under 10 requests a second. Each step resumes where it stopped if you run the cell again after a disconnect. The bank goes to Drive in section 12. `--trial` reads TRAIN years only. The real test is declared and committed in the repo first (`docs/EVENTS.md`).

In [ ]:
!python -m brain universe --band small
!python -m brain universe --band micro
!python -m brain releases --band small
!python -m brain releases --band micro
!python -m brain events declare --trial && python -m brain events test --trial

## 9. Read, decide, forecast (the model reads; the engine decides)
- **`scout`:** the model reads public pages and notes claims with their sources. It has no web search unless `BRAIN_SEARX_URL` is set.
- **`decide`:** the model makes its own call. Policy D1 weighs only evidence with a proven record, so expect NO_EDGE today.
- **`forecast`:** price ranges and the chance to beat SPY. The first run trains the live model: a few minutes on the small bank, 5–10 on the full one.

In [ ]:
!python -m brain scout AAPL --max-pages 4
!python -m brain decide {WL}
!python -m brain forecast AAPL --llm

## 10. The terminal, through Colab's port proxy
The terminal only answers its own machine. Setting `BRAIN_TERMINAL_PROXY_HOST` lets it accept the one address Colab's proxy uses; the per-session token is still required. Open the window, then type `AAPL DEC`, `AAPL FCST`, or ASK questions.

In [ ]:
from google.colab import output
from urllib.parse import urlparse
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8765)")
os.environ["BRAIN_TERMINAL_PROXY_HOST"] = urlparse(proxy_url).hostname
term = subprocess.Popen(["python", "-m", "brain", "terminal", "--no-browser", "--port", "8765"],
                        stdout=open("/content/terminal.log", "w"), stderr=subprocess.STDOUT)
time.sleep(20)
output.serve_kernel_port_as_window(8765)

## 11. Score: how did past decisions, forecasts and sources do?

In [ ]:
!python -m brain decide score
!python -m brain forecast score

## 12. Save
- The bank goes back to Drive.
- The research ledger and RESULTS.md live in the repo: push them with a token, or download them.

In [ ]:
# The web cache (cache/*.gz) is left behind: it is only a cache, and thousands of small files are slow on Drive.
!rsync -a --info=progress2 --exclude "/cache/*.gz" --exclude "/cache/*.json" "{LOCAL_VAR}/" "{DRIVE_VAR}/"
!git status --short research/
# With GITHUB_TOKEN set and something to save:
# !git add research && git commit -m "Colab run" && git push origin claude/upbeat-turing-z9jk2b

## 13. Testing the bigger model honestly
A 32B model reads and explains far better than the 1.7B one tested so far. Whether it **predicts** better is a separate question, and only its record can answer it.
- **Its record starts now.** Every `forecast --llm` run in section 9 is recorded as an F1-llm fact. Each one is scored once its sessions have passed (`forecast score`, section 11).
- **Run it weekly on the same stocks.** That builds a record on weeks the model cannot have seen.
- **The past window is used up.** F1d already holds a result for the last 8 weeks, and the code refuses a new test on a window that overlaps one with a result. A declared retest needs a new test id with a new window, written into `brain/research/forecast_tests.py` and committed before it runs.

In [ ]:
if globals().get("vllm"):
    vllm.terminate()  # stop the model server when you are done (the GPU is billed while it runs)